In [1]:
import requests
from dotenv import load_dotenv
import os


load_dotenv()

def medical_chat_completion(
    message: str,
    model: str = "medgemma-4b-it",
    max_tokens: int = 1000,
    temperature: float = 0.7,
):
    url = "https://dr7.ai/api/v1/medical/chat/completions"

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {os.environ['DR7_API_KEY']}",
    }

    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": message,
            }
        ],
        "max_tokens": max_tokens,
        "temperature": temperature,
    }

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()  # Raises an exception for 4xx/5xx responses

    return response.json()

In [3]:
result = medical_chat_completion(
    message="Patient reports headache for 3 days with fever of 38.5°C"
)

print(result)

HTTPError: 402 Client Error: Payment Required for url: https://dr7.ai/api/v1/medical/chat/completions

In [1]:
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
import os


load_dotenv()


# 1. Initialize the client with your free access token
# Replace 'your_hf_token_here' with your actual token string

HF_TOKEN = os.environ['HF_TOKEN']
# 2. Initialize the client using standard architecture
client = InferenceClient(provider="featherless-ai", token=HF_TOKEN)

# 3. Target a universally supported free-tier model
# Qwen2.5 and Llama 3.1 are permanently deployed on HF serverless infrastructure
model_id = "google/medgemma-27b-text-it"

messages = [
    {
        "role": "system",
        "content": (
            "You are an advanced clinical AI assistant. Analyze the symptoms provided, "
            "offer structured differential considerations, and state standard medical triage steps. "
            "Always append a clear disclaimer that you are an AI and not a substitute for a real doctor."
        )
    },
    {
        "role": "user",
        "content": "Patient reports headache for 3 days with a fever of 38.5°C."
    }
]

print(f"Streaming response from {model_id}...\n")

try:
    # Use client.chat_completion with stream=True
    # This natively yields simple chunk objects that contain straight text strings
    response_stream = client.chat_completion(
        model=model_id,
        messages=messages,
        max_tokens=450,
        stream=True
    )

    for chunk in response_stream:
        # Pull text safely from the choices array dictionary format
        if hasattr(chunk, 'choices') and chunk.choices:
            content = chunk.choices[0].delta.content
            if content:
                print(content, end="", flush=True)
        # Fallback if the provider returns a direct dictionary
        elif isinstance(chunk, dict) and 'choices' in chunk:
            content = chunk['choices'][0].get('delta', {}).get('content', '')
            if content:
                print(content, end="", flush=True)

except Exception as e:
    print(f"\nAn error occurred: {e}")

Streaming response from google/medgemma-27b-text-it...

Okay, I can analyze the provided symptoms and offer a structured differential diagnosis and triage recommendations based on standard medical practice.

**Patient Symptoms:**

*   Headache (3 days duration)
*   Fever (38.5°C / 101.3°F)

**Analysis & Differential Considerations:**

The combination of headache and fever suggests an underlying inflammatory or infectious process. The duration of 3 days is subacute, which helps narrow the possibilities somewhat but still leaves a broad range. Here's a structured approach to differential considerations, categorized by potential severity and system involvement:

**I. High Priority / Potentially Life-Threatening:**

*   **Meningitis (Bacterial or Viral):**
    *   *Rationale:* Headache and fever are classic signs. Bacterial meningitis is a medical emergency. Viral meningitis is generally less severe but still requires evaluation.
    *   *Associated Symptoms to Elicit:* Stiff neck (nuchal 